# Chess AlphaZero on Google Colab — Universal Game Engine

Universal Game Engine のバックエンド（Bun + gRPC）を **Colab 内で起動**し、
Python (PyTorch) の **AlphaZero 風エージェント**（Policy/Value ネット + MCTS）がチェスを自己対戦で学習します。
木探索のノード展開はすべて gRPC の `BatchSimulate` で行い、Python 側はチェスのルールを持ちません
（合法手生成・キャスリング・アンパッサン・昇格・50 手ルール・三回同形などはすべてサーバーの `ChessRuleset` が担当します）。
学習済みモデルは Google Drive に保存され、`uge_rl.serve` で実際の対局相手として使えます。

**観測 / 行動**（`ChessTensorAdapter`）: 観測は自分視点の盤面 64 マス（黒は上下反転）+ キャスリング権 4 + アンパッサン 1 + 50 手カウンタ 1 + 同形回数 1 = 71 要素で、
NN 入力では駒種ごとの one-hot などの 19 チャンネル × 8 × 8 に展開します。行動は「移動先マス × 28 種（方向 8 / ナイト 8 / 昇格 12）」= 1792 通りです。

**手順**: ランタイム → 「ランタイムのタイプを変更」で GPU (T4) を選んでから、上から順に実行してください。
オセロ版は `othello_alphazero_colab.ipynb`、将棋版は `shogi_alphazero_colab.ipynb` を参照。

## 1. Google Drive をマウント

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MODEL_DIR = '/content/drive/MyDrive/UniversalGameEngine/models'
import os; os.makedirs(MODEL_DIR, exist_ok=True)
print('models will be saved to', MODEL_DIR)

## 2. リポジトリの取得と Bun のインストール

private リポジトリの場合は `REPO_URL` を `https://<GITHUB_TOKEN>@github.com/...` の形式にしてください。

In [ ]:
import os
REPO_URL = 'https://github.com/takumi-mr/UniversalGameEngine.git'  #@param {type:"string"}
BRANCH = 'main'  #@param {type:"string"}

if not os.path.exists('/content/UniversalGameEngine'):
    !git clone --depth 1 --branch {BRANCH} {REPO_URL} /content/UniversalGameEngine
%cd /content/UniversalGameEngine

# Bun
!curl -fsSL https://bun.sh/install | bash > /dev/null 2>&1
os.environ['PATH'] = '/root/.bun/bin:' + os.environ['PATH']
!bun --version

# 依存関係（postinstall の git hook 設定は Colab では不要なのでスキップ）
!bun install --frozen-lockfile --ignore-scripts

## 3. バックエンドをバックグラウンド起動

`RL_MODE=true` にすると Redis / MongoDB なしのインメモリ動作になります。ログは `server.log` に出ます。

In [ ]:
import subprocess, sys, time
sys.path.insert(0, '/content/UniversalGameEngine/apps/ml')

env = dict(os.environ, RL_MODE='true', PORT='3000', GRPC_PORT='50051')
server = subprocess.Popen(
    ['bun', 'run', 'apps/backend/server.ts'],
    cwd='/content/UniversalGameEngine',
    env=env,
    stdout=open('/content/server.log', 'w'),
    stderr=subprocess.STDOUT,
)

from uge_rl.env import wait_for_server
wait_for_server('localhost:50051', timeout_sec=90)
print('gRPC server ready (pid', server.pid, ')')
!tail -n 5 /content/server.log

## 4. Python 依存関係

In [ ]:
!pip install -q -r apps/ml/requirements.txt
import torch; print('torch', torch.__version__, 'cuda:', torch.cuda.is_available())

## 5. 学習

1 イテレーション = 自己対戦 `--games-per-iter` 局 → `--train-steps-per-iter` 回の勾配更新。
チェスは 1 局が長いので、オセロより時間がかかります。目安として T4 で 1 イテレーション（10 局・100 探索・上限 200 手）が 10〜15 分です。

- `--max-moves`: この手数を超えた対局は引き分けとして打ち切る（50 手ルールや三回同形で終局はするが、弱いうちは長引くため）
- `--dirichlet-alpha`: ルートノイズ。チェスは AlphaZero と同じ 0.3 を使う
- 中断した場合は `--resume {MODEL_PATH}` で再開できます

In [ ]:
ITERATIONS = 30  #@param {type:"integer"}
GAMES_PER_ITER = 10  #@param {type:"integer"}
SIMULATIONS = 100  #@param {type:"integer"}
MAX_MOVES = 200  #@param {type:"integer"}
MODEL_PATH = f'{MODEL_DIR}/chess_az.pt'

!cd apps/ml && python -m uge_rl.train_az     --game chess     --address localhost:50051     --iterations {ITERATIONS}     --games-per-iter {GAMES_PER_ITER}     --simulations {SIMULATIONS}     --sim-batch 32     --max-moves {MAX_MOVES}     --dirichlet-alpha 0.3     --temp-moves 20     --channels 96 --blocks 6     --train-steps-per-iter 300     --eval-every 5 --eval-games 6 --eval-simulations 50     --out {MODEL_PATH}

## 6. 評価（ランダムプレイヤーとの対戦）

`--max-moves` を超えた対局は引き分けになります。序盤の学習ではほとんど引き分けで、学習が進むと勝ちが増えます。

In [ ]:
!cd apps/ml && python -m uge_rl.evaluate --checkpoint {MODEL_PATH} --address localhost:50051 --games 10 --max-moves {MAX_MOVES}

## 7. 学習曲線（対ランダム勝率）

In [ ]:
import json
import matplotlib.pyplot as plt

meta = json.load(open(MODEL_PATH.replace('.pt', '.json')))
hist = meta.get('eval_history', [])
if hist:
    games, wr = zip(*hist)
    plt.plot(games, wr, marker='o')
    plt.axhline(0.5, ls='--', c='gray')
    plt.xlabel('self-play games'); plt.ylabel('win rate vs random'); plt.ylim(0, 1)
    plt.title(f"Chess AlphaZero ({meta['games_played']} games, {meta['train_steps']} train steps)")
    plt.show()
print({k: meta[k] for k in ('format', 'arch', 'games_played', 'train_steps', 'saved_at', 'git_commit')})

## 8. 保存されたファイルと、モデルと対局する方法

- `chess_az.pt` — Policy/Value ネットの state_dict + メタ情報（`uge_rl.checkpoint.load_checkpoint` で復元。`format: uge-rl/az/v1`）
- `chess_az.json` — メタ情報のみ

ローカルで対局するには、`.pt` を `models/` にダウンロードしてから

```bash
task rl                                      # バックエンド（RL_MODE=true。MCTS の Simulate が使える）
cd apps/frontend && bun dev                  # フロントエンド
cd apps/ml && python -m uge_rl.serve --checkpoint ../../models/chess_az.pt   # モデルのボットサーバー
```

フロントエンドでチェスの「カスタムマッチ」→ 相手を **☁️ gRPC External** にして部屋を作ると、`serve` がその席を見つけて指し始めます。

In [ ]:
!ls -la {MODEL_DIR}

## 9. 後片付け（任意）

In [ ]:
server.terminate()
print('server stopped')